# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end template for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (see below).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', None)} \nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all the record sets available in this dataset and show a sample of their data. *All record sets and field references use their `@id`.*

In [ ]:
# List record sets and their fields using their @id
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f" - Name: {rs.get('name','')}")
    print(f" - Description: {rs.get('description','')}")
    print(" - Fields:")
    for field in rs.get('field', []):
        # Each field is either a dict or @id
        if isinstance(field, dict) and '@id' in field:
            field_id = field['@id']
        else:
            field_id = field
        print(f"    - {field_id}")
    print('')

You can also print some example records from a specific record set. Let's print up to 2 example records from each available record set.

In [ ]:
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Example records from record set {rs_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")
    print('---')

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s shown above.

In [ ]:
# Extract all available record sets into DataFrames
all_record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Failed to load dataframe for {record_set_id}: {e}")

For the analysis below, we'll focus on a main data record set. Please double-check above for actual record set `@id`s.

For this example, we will use the (likely) main record set, if present, named `cr:RecordSet/Proteins` or similar. Replace `<main_record_set_id>` below with the actual record set of interest.

In [ ]:
# Choose a main record set for example EDA
main_record_set_id = None
for rs in dataframes:
    # Often the largest record set with 'protein' or 'abundance' in name
    if 'protein' in rs.lower() or 'abundance' in rs.lower():
        main_record_set_id = rs
        break
if main_record_set_id is None and len(dataframes)>0:
    main_record_set_id = list(dataframes.keys())[0] # fallback

print(f"Main record set selected: {main_record_set_id}")
df = dataframes[main_record_set_id]
print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing fields, grouping, etc.

To do this by the schema, choose a numeric field (column) and a group/categorical field from the above columns using their `@id`s. We will search for a few example possibilities. If field IDs are descriptive, you may want to adjust these accordingly.

In [ ]:
# Suggest numeric and grouping fields
import numpy as np

print(f"Columns in main record set ({main_record_set_id}):\n{df.columns.tolist()}")

# Guess at some likely numeric fields
numeric_fields = [c for c in df.columns if ('abundance' in c.lower() or 'coverage' in c.lower() or 'count' in c.lower() or 'mw' in c.lower() or 'intensity' in c.lower())]
if not numeric_fields:
    # fallback: look for floats/ints
    for col in df.columns:
        try:
            df[col].astype(float)
            numeric_fields.append(col)
        except Exception:
            continue

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Numeric field selected: {numeric_field}")
else:
    print("No numeric field found, using the first column.")
    numeric_field = df.columns[0]

# Grouping field - try to pick a reasonable one
group_fields = [c for c in df.columns if 'group' in c.lower() or 'type' in c.lower() or 'species' in c.lower() or 'sample' in c.lower()]
if group_fields:
    group_field = group_fields[0]
else:
    group_field = None

# Ensure numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75) if not np.isnan(df[numeric_field].quantile(0.75)) else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"Boxplot of {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to inspect the FAIR^2 mass spectrometry dataset and explored its record sets and fields. By selecting fields with meaningful IDs, filtering for high-abundance protein entries, normalizing values, and visualizing distributions, we produced data ready for analysis or downstream ML workflows.

You can adapt this notebook to deeper domain-specific questions, e.g., focusing on specific protein types, modifications, or subsetting by experiment.

**All data references are made using the Croissant `@id` for transparency and reproducibility.**